# Preprocesamiento y Reduccion de Dimensionalidad con PCAAnalisis de Riesgo Geologico Global: pipeline completo de preprocesamiento, PCA y visualizacion geoespacial.

In [1]:
import sys, os, warningswarnings.filterwarnings('ignore')project_root = os.path.join(os.getcwd(), '..')sys.path.insert(0, os.path.abspath(project_root))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom src.data_loader import cargar_datos_combinadosfrom src.preprocessing import pipeline_preprocesamiento_pca, cargar_pipeline, detectar_columnas_skewfrom src.visualization import (    plot_varianza_acumulada, plot_pca_2d, plot_biplot, plot_pca_interactivo,    plot_mapa_riesgo, plot_pairplot_pca, plot_loadings_heatmap, plot_pca_3d,)sns.set_theme(style='whitegrid', context='paper', font='sans-serif', font_scale=1.2, palette='mako')plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})os.makedirs('figures', exist_ok=True)print('Librerias cargadas correctamente')

Librerias cargadas correctamente


## 1. Ingesta de datos geologicosCarga de datos combinados desde fuentes sismologicas, ciclones tropicales, volcanes y datos auxiliares. Los datos son sinteticos para desarrollo del pipeline; la ingesta final se integrara con H3, USGS, IBTrACS y GVP.

In [1]:
df_raw = cargar_datos_combinados()print(f'Dimensiones: {df_raw.shape}')print(f'Columnas: {list(df_raw.columns)}')print(f'Rango lat: [{df_raw['lat'].min():.1f}, {df_raw['lat'].max():.1f}]')print(f'Rango lon: [{df_raw['lon'].min():.1f}, {df_raw['lon'].max():.1f}]')df_raw.head()

Dimensiones: (500, 10)
Columnas: ['magnitud_max_sismo', 'profundidad_media_sismo', 'frecuencia_eventos_sismicos', 'viento_max_ciclones', 'presion_min_ciclones', 'elevacion_volcan', 'categoria_tormenta', 'tipo_volcan', 'lat', 'lon']
Rango lat: [-60.0, 59.9]
Rango lon: [-179.7, 179.1]


## 2. Analisis exploratorio de features numericasEstadisticos descriptivos y deteccion de asimetria (skewness) en las variables geologicas.

In [1]:
num_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()excluir = {'lat', 'lon', 'h3_index', 'cell_id'}features = [c for c in num_cols if c not in excluir]df_stats = df_raw[features].describe().Tdf_stats['skewness'] = df_raw[features].skew()df_stats['skew_alto'] = df_stats['skewness'].abs() > 0.75df_stats[['mean', 'std', 'skewness', 'skew_alto']]

## 3. Transformacion logaritmica (log1p)Aplicamos log1p a columnas con skew alto para reducir asimetria y estabilizar la varianza antes del escalado.

In [1]:
skew_cols = detectar_columnas_skew(df_raw, umbral=0.75)print(f'Columnas con skew alto (>0.75): {skew_cols}')df_log = df_raw.copy()for col in skew_cols:    df_log[f'{col}_log'] = np.log1p(df_log[col].clip(lower=0))skew_after = df_log[[f'{c}_log' for c in skew_cols if f'{c}_log' in df_log.columns]].skew()print('\nSkew despues de log1p:')print(skew_after.round(3))

Columnas con skew alto (>0.75): ['magnitud_max_sismo', 'profundidad_media_sismo', 'viento_max_ciclones', 'elevacion_volcan']

Skew despues de log1p:
magnitud_max_sismo_log         0.829
profundidad_media_sismo_log    0.413
viento_max_ciclones_log        0.298
elevacion_volcan_log          -0.060
dtype: float64


## 4. One-Hot Encoding y StandardScalerCodificamos variables categoricas y estandarizamos (media=0, varianza=1) para garantizar que PCA no se sesgue por diferencias de escala entre variables.

In [1]:
cat_cols = ['categoria_tormenta', 'tipo_volcan']dummy_cols = [c for c in cat_cols if c in df_log.columns]df_encoded = pd.get_dummies(df_log, columns=dummy_cols, drop_first=True)exclude_final = excluir | {'volcano_name', 'region', 'country'}cols_numeric = [c for c in df_encoded.select_dtypes(include=[np.number]).columns if c not in exclude_final]from sklearn.preprocessing import StandardScalerscaler = StandardScaler()X_scaled = scaler.fit_transform(df_encoded[cols_numeric])df_scaled = pd.DataFrame(X_scaled, columns=cols_numeric, index=df_raw.index)print(f'Shape escalado: {df_scaled.shape}')pd.DataFrame({'media': df_scaled.mean().round(6), 'std': df_scaled.std().round(6)}).head()

Shape escalado: (500, 10)


## 5. Reduccion de dimensionalidad con PCAAplicamos Analisis de Componentes Principales para capturar la maxima varianza posible en pocos componentes ortogonales.

In [1]:
from sklearn.decomposition import PCApca = PCA(random_state=42)X_pca = pca.fit_transform(X_scaled)explained_var = pca.explained_variance_ratio_cum_var = np.cumsum(explained_var)n_85 = int(np.searchsorted(cum_var, 0.85) + 1)print(f'Varianza explicada por PC1: {explained_var[0]*100:.1f}%')print(f'Varianza explicada por PC2: {explained_var[1]*100:.1f}%')print(f'Componentes para 85%% varianza: {n_85}')print(f'Varianza acumulada con {n_85} comps: {cum_var[n_85-1]*100:.1f}%')df_pca = pd.DataFrame(    X_pca[:, :n_85],    columns=[f'PC{i+1}' for i in range(n_85)],    index=df_raw.index)df_pca.head()

Varianza explicada por PC1: 20.8%
Varianza explicada por PC2: 20.1%
Componentes para 85%% varianza: 5
Varianza acumulada con 5 comps: 86.4%


### 5.1 Varianza explicada acumulada

In [1]:
plot_varianza_acumulada(pca, threshold=0.85, save_path='figures/varianza_acumulada.png')plt.show()

TypeError: 'list' object is not a mapping

### 5.2 Proyeccion 2D

In [1]:
plot_pca_2d(df_pca, pca_model=pca, save_path='figures/pca_2d.png')plt.show()

TypeError: 'list' object is not a mapping

### 5.3 Biplot: contribucion de variables originales

In [1]:
plot_biplot(df_pca, pca_model=pca, feature_names=cols_numeric, save_path='figures/biplot.png')plt.show()

TypeError: 'list' object is not a mapping

### 5.4 PCA 3D Interactivo (Plotly)

In [1]:
plot_pca_3d(df_pca, pca_model=pca, save_path='figures/pca_3d.html')print('Grafico 3D guardado en figures/pca_3d.html')

Grafico 3D guardado en figures/pca_3d.html


## 6. Loadings: que variables contribuyen a cada componente?El heatmap muestra que variables geologicas (sismicidad, ciclones, volcanes) tienen mayor peso en cada componente principal.

In [1]:
plot_loadings_heatmap(pca, feature_names=cols_numeric, n_components=n_85, save_path='figures/loadings_heatmap.png')plt.show()

TypeError: 'list' object is not a mapping

## 7. Pairplot de componentes principales

In [1]:
plot_pairplot_pca(df_pca, n_components=min(4, n_85), save_path='figures/pairplot_pca.png')plt.show()

TypeError: 'list' object is not a mapping

## 8. Mapa geografico de riesgo (Folium)Cada punto representa una celda geografica. El color indica el valor de PC1 (mayor = senal geologica mas extrema). Popups muestran magnitud sismica, viento de ciclones y elevacion volcanica.

In [1]:
df_mapa = df_raw[['lat', 'lon', *features]].copy()df_mapa['PC1'] = df_pca['PC1'].valuesplot_mapa_riesgo(    df_mapa,    lat_col='lat', lon_col='lon',    color_col='PC1',    popup_cols=['magnitud_max_sismo', 'viento_max_ciclones', 'elevacion_volcan'],    radius_scale=2.0,    save_path='figures/mapa_riesgo.html')print('Mapa interactivo guardado en figures/mapa_riesgo.html')

Mapa interactivo guardado en figures/mapa_riesgo.html


## 9. Grafico interactivo 2D (Plotly)

In [1]:
plot_pca_interactivo(df_pca, pca_model=pca, save_path='figures/pca_interactivo.html')print('HTML guardado en figures/pca_interactivo.html')

HTML guardado en figures/pca_interactivo.html


## 10. Pipeline completo exportable a produccionConstruimos un Pipeline sklearn con SkewLogTransformer, OneHotTransformer, StandardScaler y PCA. Se exporta con joblib para reutilizacion en produccion.

In [1]:
df_pipe, pipeline, _ = pipeline_preprocesamiento_pca(    df_raw, target_variance=0.85, save_path='models/pipeline_riesgo.pkl')print(f'Pipeline exportado: {df_pipe.shape[1]} componentes')print(f'Varianza explicada: {pipeline.named_steps['pca'].explained_variance_ratio_.cumsum()[-1]:.2%}')loaded = cargar_pipeline('models/pipeline_riesgo.pkl')X_test = loaded.transform(df_raw.head(5))print(f'Transform sobre 5 muestras: {X_test.shape}')

Pipeline exportado: 5 componentes
Varianza explicada: 86.42%
Transform sobre 5 muestras: (5, 5)


## 11. Guardado de datos procesados

In [1]:
df_pca.to_csv('data/processed/dataset_pca.csv', index=False)df_scaled.to_csv('data/processed/dataset_scaled.csv', index=False)print('Datos guardados en data/processed/')

Datos guardados en data/processed/
